# Lexington, Missouri — Land Value Tax Model

**City of Lexington, Lafayette County, Missouri** — 4:1 split-rate LVT model, city general-revenue levy only, preserving all exemptions.

Parcel source: **Regrid** ("Data with Purpose", CLE licensed), Lafayette County geoparquet refreshed 2025-04-15, **clipped to the City of Lexington municipal limits** (Census place **2941870**, "Lexington city", 3,346 land acres). The pre-clipped input is `data/regrid_lexington_cityclip.parquet` (gitignored — licensed data, never committed); the clip boundary is `data/city_boundary_2941870.geojson`.

**Boundary change (2026-07-27 re-run).** The prior run used the postal `scity == 'LEXINGTON'` filter, which pulled in 1,102 rural parcels lying outside the municipal limits that do not pay the city levy. Greg approved re-running against the true city-limits clip. This notebook loads the pre-clipped city-limits file directly.

**Revenue-neutrality anchor change (2026-07-27 methodology decision, Greg).** The certified $301,171 / $54.9M general-revenue figure is a **real + personal property aggregate** (Missouri taxes personal property — vehicles/equipment — at 33⅓%; it is ~36% of this roll). Anchoring that full aggregate levy onto a real-property-only parcel file forced the calibration factor to K ≈ 1.57 and inflated every absolute per-parcel bill ~1.57×. This run instead anchors revenue-neutrality to the **real-property-only share** of the levy: the raw city-limits real-property assessed base at the statutory rate (**~$35.1M × 5.481 mills ≈ $192.4k**). By construction this drives **K ≈ 1.0**, so absolute per-parcel bills now reflect what real property actually pays. The revenue-neutral **split millages are unchanged** (they are level-independent ratios); only absolute dollars deflate (by ~1/1.5652). Gate 1 now reconciles to the **real-property-only target (~$192.4k)**, not $301,171 — this is intended.

Follows the LVT Shift canonical 7-section notebook template.

## Section 1 — Imports and Setup

In [ ]:
import sys
import os
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)

# Constants
CITY_NAME = 'lexington_mo'
STATE_FIPS = '29'              # Missouri
COUNTY_FIPS = '107'            # Lafayette County
MODEL_TYPE = 'split_rate_4to1'
LAND_IMPROVEMENT_RATIO = 4.0

# Parcel-map identity columns (Regrid schema)
PARCEL_ID_COL = 'parcelnumb'
OWNER_NAME_COL = 'owner'
OWNER_ADDRESS_COL = 'address'
PARCEL_URL_TEMPLATE = None     # Lafayette County has no stable public per-parcel deep link

# Missouri statutory assessment ratios (RSMo 137.115) applied to market value
RATIO_RESIDENTIAL = 0.19
RATIO_COMMERCIAL = 0.32
RATIO_AGRICULTURAL = 0.12

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

## Section 2 — Load / Build Parcel Data

The staged input is the Lafayette County Regrid file **already clipped to the City of Lexington municipal limits** (Census place 2941870): `data/regrid_lexington_cityclip.parquet`, 2,521 rows / 2,467 unique parcels. This replaces the prior run's county-wide load + postal `scity == 'LEXINGTON'` filter, which had over-included 1,102 rural parcels outside the city.

We:
1. load the pre-clipped city-limits file (no `scity` filter needed — the clip is the municipal boundary),
2. **dedupe stacked geometries**: some parcels appear as several rows sharing `parcelnumb` with different geometry fragments but **identical** `landval`/`improvval`/`parval` (the value is a parcel attribute repeated on each fragment — summing would triple-count). We keep one record per `parcelnumb`, unioning the geometry fragments,
3. cache the result to `data/parcels.gpq` (the canonical per-parcel cache that downstream stages read). The cache is rebuilt unconditionally on this run so the stale postal-run cache is overwritten.

In [ ]:
PARCEL_PATH = DATA_DIR / 'parcels.gpq'
CITYCLIP_PATH = DATA_DIR / 'regrid_lexington_cityclip.parquet'

# Pre-clipped City-of-Lexington universe (Census place 2941870). Rebuild the
# canonical parcels.gpq cache unconditionally so any stale postal-run cache is replaced.
lex = gpd.read_parquet(CITYCLIP_PATH).rename_geometry('geometry')
print(f"City-clip raw rows (pre-dedupe): {len(lex):,}  unique parcelnumb: {lex['parcelnumb'].nunique():,}")

# Dedupe stacked duplicate geometries on parcelnumb; union geometry fragments.
dup_ids = lex['parcelnumb'][lex['parcelnumb'].duplicated(keep=False)].unique()
singles = lex[~lex['parcelnumb'].isin(dup_ids)].copy()
dgroups = lex[lex['parcelnumb'].isin(dup_ids)].copy()
dissolved = dgroups.dissolve(by='parcelnumb', aggfunc='first', as_index=False)
gdf = pd.concat([singles, dissolved], ignore_index=True)
gdf = gpd.GeoDataFrame(gdf, geometry='geometry', crs=lex.crs).copy()

gdf.to_parquet(PARCEL_PATH)
print(f"Deduped to {len(gdf):,} unique parcels ({len(dup_ids)} had stacked geometry); cached to {PARCEL_PATH}")

# Numeric value fields
for c in ['landval', 'improvval', 'parval', 'agval', 'll_gisacre']:
    gdf[c] = pd.to_numeric(gdf[c], errors='coerce').fillna(0.0)
print("geometry valid:", gdf.geometry.is_valid.mean(), " CRS:", gdf.crs)
print("built (improvval>0): {:,}   vacant (improvval==0): {:,}".format(
    int((gdf['improvval'] > 0).sum()), int((gdf['improvval'] <= 0).sum())))
print("market value sums — land ${:,.0f}  improvement ${:,.0f}  parcel ${:,.0f}".format(
    gdf['landval'].sum(), gdf['improvval'].sum(), gdf['parval'].sum()))

## Section 3 — Classify and Validate

**Exemptions (preserved).** Regrid carries no assessor exempt flag for this county (`taxamt`, `homestead_exemption`, `owntype` are all null on the Lexington subset). We infer the Missouri constitutional exemptions (Art. X §6 — government, religious, educational/charitable) from `lbcs_ownership_desc`. These parcels are held out of the taxable base and the reform, exactly as the assessor's certified valuation excludes them.

**Assessment class → Missouri ratio.** Regrid values are `parvaltype == 'MARKET'`, i.e. market value, **not** Missouri assessed value. Missouri assesses by subclass (RSMo 137.115): residential 19 %, commercial 32 %, agricultural 12 %. We assign each taxable parcel a class from the strongest available signals (`lbcs_ownership_desc`, `zoning`/`zoning_type`, `lbcs_structure_desc`, presence of a building):

- **Agricultural (12 %)** — AG-zoned, unimproved, > 5 acres (genuine farm/open land; excludes AG-zoned house lots).
- **Commercial (32 %)** — business-entity ownership, or commercial/industrial zoning.
- **Residential (19 %)** — everything else (dwellings, in-town house and vacant lots).

`usecode`/`usedesc` are entirely null on this subset, so the class comes from the LBCS + zoning signals above rather than an assessor use code.

In [ ]:
own = gdf['lbcs_ownership_desc']
zon = gdf['zoning']
ztype = gdf['zoning_type']
struct = gdf['lbcs_structure_desc']
has_bldg = gdf['improvval'] > 0

EXEMPT_OWNERS = ['City, Village, Township, etc.', 'State government', 'Federal government',
                 'County, Parish, Province, etc.', 'Nonprofit religious', 'Nonprofit educational']
gdf['full_exmp'] = own.isin(EXEMPT_OWNERS).astype(int)
exempt = gdf['full_exmp'] == 1

is_ag = (~exempt) & (zon == 'AG') & (~has_bldg) & (gdf['ll_gisacre'] > 5)
is_comm = (~exempt) & (~is_ag) & ((own == 'Businesses and commercial entities') | ztype.isin(['Commercial', 'Industrial']))
is_res = (~exempt) & (~is_ag) & (~is_comm)

gdf['assess_ratio'] = np.select([exempt, is_ag, is_comm, is_res],
                                 [0.0, RATIO_AGRICULTURAL, RATIO_COMMERCIAL, RATIO_RESIDENTIAL],
                                 default=RATIO_RESIDENTIAL)
gdf['assess_class'] = np.select([exempt, is_ag, is_comm, is_res],
                                 ['exempt', 'agricultural', 'commercial', 'residential'], default='residential')
print(gdf['assess_class'].value_counts())

# Property category for the report
cat = pd.Series('Other', index=gdf.index)
cat[is_res] = 'Single Family Residential'
cat[is_comm] = 'Commercial'
cat[is_ag] = 'Agricultural'
cat[exempt] = 'Exempt'
gdf['PROPERTY_CATEGORY'] = cat
# Mandatory override: $0 improvement -> Vacant Land (catches empty lots incl. unimproved ag land)
gdf.loc[(gdf['improvval'] <= 0) & (~exempt), 'PROPERTY_CATEGORY'] = 'Vacant Land'
print(gdf.loc[~exempt, 'PROPERTY_CATEGORY'].value_counts())

## Section 4 — Current Tax Model + Revenue Gate

**Levy.** City of Lexington general-revenue municipal real-property levy: **$0.5481 per $100** of assessed value = **5.481 mills** (per $1,000). Source: Missouri State Auditor, *2025 Property Tax Rates*, Report No. 2026-006 (Jan 2026), City of Lexington / General Revenue — certified assessed valuation **$54,948,146** × $0.5481/$100 = **$301,171** levied. **That certified figure is a real + personal property aggregate** and is kept here only as a reference control, not as the revenue-neutrality anchor (see below).

**Market → assessed.** Regrid values are `parvaltype == 'MARKET'`, i.e. market value, **not** Missouri assessed value. Missouri assesses by subclass (RSMo 137.115): residential 19 %, commercial 32 %, agricultural 12 %. We assign each taxable parcel a class from the strongest available signals (`lbcs_ownership_desc`, `zoning`/`zoning_type`, `lbcs_structure_desc`, presence of a building). On the **city-limits clip** the resulting real-property assessed base totals **~$35.1M**.

**Revenue-neutrality anchor = the real-property-only levy (methodology decision, Greg, 2026-07-27).** The certified $301,171 / $54.9M aggregate includes **personal property** (vehicles, business equipment, assessed at 33⅓%) and state-assessed utilities that no real-property parcel file can contain — roughly 36% of this roll. Anchoring the *full* aggregate levy onto the real-property-only base forced `K = 54.9M / 35.1M ≈ 1.5652` and inflated every absolute per-parcel bill ~1.57×. For a council deck we want realistic absolute bills, so we now anchor to the **real-property-only share** of the levy:

> **target revenue = (city-limits real-property assessed base) × statutory rate = ~$35.1M × 5.481 mills ≈ $192,414.**

This is what real property currently pays under the flat rate. It is self-consistent by construction, so the uniform calibration factor is **K = real_property_AV / raw_base = 1.0000** — no personal-property padding. The taxable base is therefore the raw real-property assessed base itself.

**K = 1.0 is level-only and cancels from the reform.** As before, `K` is uniform across land and improvements, so it **cancels out of the revenue-neutral split-rate solution**: the land/improvement millages and every per-parcel *percentage* change are identical regardless of the anchor. Switching the anchor from the aggregate levy (K=1.57) to the real-property-only levy (K=1.0) leaves the reform's distributional results **unchanged** and deflates only the *absolute* dollar figures (by ~1/1.5652 ≈ 0.639) to realistic levels. **Gate 1 now reconciles the modeled current revenue to the ~$192.4k real-property target, not $301,171 — intended.**

In [ ]:
RATE_PER_1000 = 5.481          # mills (per $1,000 assessed)

# Reference-only aggregates from the auditor (real + personal + state-assessed).
# These are NOT the revenue-neutrality anchor for this run — see the methodology note.
CERTIFIED_AV_AGGREGATE = 54_948_146   # MO State Auditor 2026-006, City of Lexington General Revenue (real+personal)
CERTIFIED_LEVY_AGGREGATE = 301_171    # = CERTIFIED_AV_AGGREGATE x rate (real+personal aggregate levy)

# Assessed (market x subclass ratio) before calibration
gdf['assessed_land_raw'] = gdf['landval'].clip(lower=0) * gdf['assess_ratio']
gdf['assessed_improvement_raw'] = gdf['improvval'].clip(lower=0) * gdf['assess_ratio']

raw_base = (gdf.loc[~exempt, 'assessed_land_raw'] + gdf.loc[~exempt, 'assessed_improvement_raw']).sum()

# Revenue-neutrality anchor (methodology decision, Greg, 2026-07-27): anchor to the
# REAL-PROPERTY-ONLY share of the city levy = the raw city-limits real-property
# assessed base at the statutory rate. NOT the real+personal aggregate. Self-consistent
# by construction, so K == 1.0 (no personal-property padding); absolute per-parcel bills
# then reflect what real property actually pays under the flat rate.
REAL_PROPERTY_AV = raw_base
OFFICIAL_REVENUE = REAL_PROPERTY_AV * RATE_PER_1000 / 1000.0   # real-property-only target
K = REAL_PROPERTY_AV / raw_base                                # == 1.0 by construction

print(f"Raw taxable assessed base (Regrid x ratios):     ${raw_base:,.0f}")
print(f"Real-property-only revenue target (base x rate): ${OFFICIAL_REVENUE:,.0f}")
print(f"Reference aggregate levy (real+personal):        ${CERTIFIED_LEVY_AGGREGATE:,.0f}")
print(f"Uniform calibration factor K = {K:.4f}")

gdf['taxable_land_value'] = gdf['assessed_land_raw'] * K
gdf['taxable_improvement_value'] = gdf['assessed_improvement_raw'] * K
gdf['taxable_value'] = gdf['taxable_land_value'] + gdf['taxable_improvement_value']
gdf['millage_rate'] = RATE_PER_1000

current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='taxable_value',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

gap_pct = (current_revenue / OFFICIAL_REVENUE - 1) * 100
print(f"\nModeled current revenue: ${current_revenue:,.0f}   Real-property target: ${OFFICIAL_REVENUE:,.0f}   Gap: {gap_pct:+.2f}%")
assert abs(gap_pct) < 5.0, f"Revenue gap {gap_pct:.1f}% exceeds threshold"
print("Revenue gate (Gate 1): PASS — reconciles to the real-property-only target")

## Section 5 — Split-Rate Model (4:1)

Land taxed at 4× the improvement rate, revenue-neutral to the city general-revenue levy. Fully-exempt parcels are held out of the solver and dropped from the modeled output (they carry no signal), and reported by count only.

In [ ]:
n_exempt = int((gdf['full_exmp'] == 1).sum())
gdf = gdf[gdf['full_exmp'] == 0].copy()

land_millage, improvement_millage, new_revenue, gdf = model_split_rate_tax(
    df=gdf,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=gdf['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

print(f"Held out {n_exempt:,} fully-exempt parcels (excluded from model, export, and charts).")
print(f"Land millage: {land_millage:.4f}   Improvement millage: {improvement_millage:.4f}   (current flat: {RATE_PER_1000:.4f})")
print(f"Revenue check: ${new_revenue:,.0f}")

category_summary = calculate_category_tax_summary(
    df=gdf, category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax', new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title=f"{CITY_NAME} — 4:1 Split-Rate Tax Impact")

### Artifact scan (Gate 5)

Read the category table against economic priors before trusting it.

In [ ]:
d = gdf.copy()
d['bldg_share'] = d['taxable_improvement_value'] / (
    d['taxable_land_value'] + d['taxable_improvement_value']).replace(0, np.nan)
scan = d.groupby('PROPERTY_CATEGORY').agg(
    n=('tax_change_pct', 'size'),
    median_pct=('tax_change_pct', 'median'),
    median_bldg_share=('bldg_share', 'median'),
    pct_zero_bldg=('taxable_improvement_value', lambda s: (s <= 1000).mean()),
).sort_values('median_pct', ascending=False)
print(scan.round(3))

**Gate 5 reading.** Vacant Land shows the large increase expected of underused land; built categories (SFR, Commercial) show modest decreases with plausible ~0.8 building shares — no ceiling clustering, no distinct categories pinned at one extreme value, no $0-building placeholder cluster. Land is a low share of total value (~16-18%), which is economically real for a small rural county-seat where structures dominate parcel value; it is not an artifact. Result is believable.

In [ ]:
# Representative single-family-home bill (absolute-level sanity check under K=1.0)
sfr = gdf[gdf['PROPERTY_CATEGORY'] == 'Single Family Residential'].copy()
med_land = sfr['taxable_land_value'].median()
med_tot = sfr['taxable_value'].median()
med_cur = sfr['current_tax'].median()
med_new = sfr['new_tax'].median()
print(f"Median SFR taxable assessed value: ${med_tot:,.0f}  (land ${med_land:,.0f})")
print(f"Median SFR current city bill: ${med_cur:,.2f}   ->  4:1 split-rate bill: ${med_new:,.2f}"
      f"   (change {(med_new/med_cur-1)*100:+.1f}%)")
print(f"Mean   SFR current city bill: ${sfr['current_tax'].mean():,.2f}   ->  ${sfr['new_tax'].mean():,.2f}")

# Pick the parcel closest to the median total value as a concrete representative home
rep = sfr.iloc[(sfr['taxable_value'] - med_tot).abs().argsort().iloc[0]]
print(f"\nRepresentative home (parcel {rep.get('parcelnumb','?')}):")
print(f"  assessed value ${rep['taxable_value']:,.0f}  (land ${rep['taxable_land_value']:,.0f} / "
      f"improvement ${rep['taxable_improvement_value']:,.0f})")
print(f"  current city bill ${rep['current_tax']:,.2f}  ->  split-rate ${rep['new_tax']:,.2f}  "
      f"({rep['tax_change_pct']:+.1f}%)")

## Section 6 — Exploration Chart (optional)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
gdf.groupby('PROPERTY_CATEGORY')['tax_change_pct'].median().sort_values().plot.barh(ax=ax)
ax.set_title(f'{CITY_NAME} — Median Tax Change % by Category (4:1 Split-Rate)')
ax.set_xlabel('Median % Change')
ax.axvline(0, color='k', lw=0.8)
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=150)
plt.close()
print('saved category_preview.png')

## Section 7 — Census Join + Export (canonical closing pattern)

In [ ]:
# Census join — must happen before export
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
        except concurrent.futures.TimeoutError:
            print("Census API timed out — skipping census join")
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f"Census join failed: {e}")
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

In [ ]:
# Export — gdf must have census columns by this point
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

# Standard report — PNGs in analysis/reports/<city>/
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)

# Parcel-map export — GeoParquet + interactive HTML map
from lvt.parcel_map import save_parcel_map_export, create_parcel_map
map_gdf = save_parcel_map_export(
    gdf=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/maps/{CITY_NAME}.parquet',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    parcel_id_col=PARCEL_ID_COL,
    parcel_url_template=PARCEL_URL_TEMPLATE,
    owner_name_col=OWNER_NAME_COL,
    owner_address_col=OWNER_ADDRESS_COL,
)
create_parcel_map(map_gdf, CITY_NAME)
print("Done.")

## Limitations & Data Notes

- **Parcel source:** Regrid (licensed "Data with Purpose"), Lafayette County, refreshed 2025-04-15, **clipped to the City of Lexington municipal limits** (Census place 2941870, "Lexington city"). Input `data/regrid_lexington_cityclip.parquet`; boundary `data/city_boundary_2941870.geojson`. Gitignored — never committed.
- **Boundary basis (changed 2026-07-27):** prior run filtered the county file by postal `scity == 'LEXINGTON'` (3,569 unique parcels), which over-included **1,102 rural parcels** outside the municipal limits that pay no city levy. This run uses the true city-limits clip: **2,521 raw rows → 2,467 unique parcels** (47 stacked-geometry ids unioned).
- **Market → assessed:** Regrid values are `MARKET`. Converted to Missouri assessed value via statutory subclass ratios (residential 19 %, commercial 32 %, agricultural 12 %; RSMo 137.115). Subclass inferred from `lbcs_ownership_desc` / `zoning` / `lbcs_structure_desc` because `usecode`/`usedesc` are null on this subset.
- **Revenue-neutrality anchor = real-property-only levy (methodology decision, Greg, 2026-07-27):** the auditor's certified $54,948,146 / $301,171 general-revenue figure is a **real + personal + state-assessed aggregate**; Missouri personal property (vehicles/equipment at 33⅓ %) is typically ~30–45 % of a small city's roll — value no real-property parcel file can contain. Anchoring that full aggregate onto the real-property-only base forced K ≈ 1.57 and inflated absolute bills ~1.57×. This run instead anchors to the **real-property-only share** of the levy: the raw city-limits real-property assessed base (~$35.1M) × the statutory rate (5.481 mills) ≈ **$192.4k**. This is self-consistent, so **K = 1.0** (no personal-property padding), and absolute per-parcel bills reflect what real property actually pays under the flat rate.
- **What did / didn't change vs. the K≈1.57 run:** because K is uniform, it cancels from the reform — the **split-rate millages (~land 14.94 / improvement 3.74) and every per-parcel % change are identical**. Only the *absolute* dollar figures changed: modeled base, modeled levy, and every per-parcel bill deflate by ~1/1.5652 ≈ 0.639. **Gate 1 now reconciles to the ~$192.4k real-property-only target, not $301,171** — intended.
- **Exemptions:** no assessor exempt flag in the data; government / religious / educational parcels inferred from ownership type and held out (preserving existing exemptions).
- **Agricultural land:** Missouri assesses farmland on *productive* value, which the market-based Regrid values only approximate. On the city clip only 2 genuine unimproved farm parcels remain; any residual unimproved parcels fold into the "Vacant Land" report category by the standard $0-improvement override.